# Regional Observation
What are the top amenity characteristics for listings in different regions?

In [ ]:
import pandas as pd
import json, re

berlin = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_berlin_listings.csv')
munich = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_munich_listings.csv')

In [ ]:
def parse_amenities(raw):
    if pd.isna(raw): return []
    try: return json.loads(raw.replace('""', '"'))
    except:
        m = re.findall(r'"([^"]+)"', raw)
        return m if m else []

def top_amenities(df, city, n=10):
    df = df.copy()
    df['price'] = df['price'].replace('[\€\$,]', '', regex=True).astype(float, errors='ignore')
    df = df[pd.to_numeric(df['price'], errors='coerce') > 0]
    df['rating'] = pd.to_numeric(df['review_scores_rating'], errors='coerce')
    df['revenue'] = pd.to_numeric(df['estimated_revenue_l365d'], errors='coerce').fillna(0)
    
    stats = {}
    for _, row in df.iterrows():
        for a in parse_amenities(row.get('amenities', '')):
            a = a.strip()
            if not a: continue
            if a not in stats: stats[a] = {'rev': 0, 'rat': 0, 'count': 0, 'rat_count': 0}
            stats[a]['rev'] += row['revenue']
            stats[a]['count'] += 1
            if pd.notna(row['rating']):
                stats[a]['rat'] += row['rating']
                stats[a]['rat_count'] += 1
    
    total = len(df)
    results = []
    for name, s in stats.items():
        if s['count'] > total * 0.05 and s['count'] < total * 0.5 and s['rat_count'] > 0 and (s['rat']/s['rat_count']) >= 4.8:
            results.append({'Amenity': name, 'Avg Rating': round(s['rat']/s['rat_count'], 2), 'Avg Revenue': round(s['rev']/s['count']), 'Listings': s['count']})
    
    result = pd.DataFrame(results).sort_values('Avg Revenue', ascending=False).head(n).reset_index(drop=True)
    result.index += 1
    print(f'--- {city}: Top {n} Amenities (rating >= 4.8, ranked by revenue) ---')
    print(result)
    print()

top_amenities(berlin, 'Berlin')
top_amenities(munich, 'Munich')